## 6. Agent loop 工作原理

> 来源：[How the agent loop works](https://code.claude.com/docs/en/agent-sdk/agent-loop)

上一节跑通了，这一节讲清楚 `async for` 背后那个循环在做什么。


### 6.1 循环五步

1. **收 prompt**：Claude 收到 prompt + system prompt + tool 定义 + 对话历史。SDK yield 一条 `subtype="init"` 的 `SystemMessage`（含 session 元数据）。
2. **评估并响应**：Claude 可能回文本、请求一个或多个 tool call、或两者都有。SDK yield `AssistantMessage`。
3. **执行 tools**：SDK 执行每个请求的 tool 并把结果回灌给 Claude（以 `UserMessage` 形式出现在流里）。hooks 可以在执行前拦截/修改/阻断。
4. **重复**：步骤 2–3 循环。**一个 turn = 一个完整的「模型响应 → 执行 tool → 结果回灌」周期**；`max_turns` 只计带 tool call 的 turn。
5. **收尾**：Claude 产出不含 tool call 的纯文本响应后，SDK yield 最终 `AssistantMessage`，随后是 `ResultMessage`（最终文本、token 用量、费用、session ID）。

这五步里的概念有大小之分，从外到内套三层：一次 `query()` 就是一个 session，对应一整个 agent loop；循环每转一圈是一个 turn（定义见第 4 步）；每个 turn 内 SDK 至少向 API 发一次请求。API 无状态，每次请求都把全部内容重发一遍（§6.3「context window」）。

五步对应到 §5「最小可运行示例」那次真实运行的消息流（`num_turns=4`，共 7 条 `AssistantMessage`，节选）：

In [ ]:
# 第 1 步：init
SystemMessage(subtype='init', data={'session_id': 'd2acdb69-5f5d-42f3-bb06-478a711fa1a2', 'tools': ['Task', 'TaskOutput', 'Bash', 'Glob', 'Grep', 'Read', ...], ...})

# 第 2 步：先回了一句文本，又请求 tool call——两者各是一条 AssistantMessage
AssistantMessage(content=[TextBlock(text="I'll explore the project structure to identify and summarize the main modules.")], ...)
AssistantMessage(content=[ToolUseBlock(id='toolu_01NChTxTWgMJq5zuLk6SQKmS', name='Bash', input={'command': 'pwd', ...})], ...)

# 第 3 步：SDK 执行 pwd，结果以 UserMessage 回灌，tool_use_id 对回上面那次调用
UserMessage(content=[ToolResultBlock(tool_use_id='toolu_01NChTxTWgMJq5zuLk6SQKmS', content='/Users/liangzhu/Library/.../claude-agent-sdk', is_error=False)], ...)

# 第 4 步：ls -la、读文件…… 2–3 步又循环了三个 turn（略）

# 第 5 步：纯文本收尾 + ResultMessage
AssistantMessage(content=[TextBlock(text='Based on my reading of the project, this is a **comprehensive Jupyter notebook tutorial** ...')], ...)
ResultMessage(subtype='success', is_error=False, num_turns=4, total_cost_usd=0.13669605, result='Based on my reading of the project, ...')



简单问题 1–2 turn；复杂任务可跨几十个 tool call。**开放式 prompt（"improve this codebase"）可能跑很久，生产 agent 默认就该设 `max_turns` / `max_budget_usd`。**


### 6.2 内置 tools 全景

| 类别 | Tools | 作用 |
|---|---|---|
| 文件操作 | `Read`, `Edit`, `Write` | 读、改、建文件 |
| 搜索 | `Glob`, `Grep` | 按模式找文件、正则搜内容 |
| 执行 | `Bash`, `Monitor` | shell 命令、脚本、git；`Monitor` 监视后台脚本或 `ws` WebSocket 源，把每行输出当事件响应（可配 `persistent` 持续监视、`timeout_ms` 超时） |
| Web | `WebSearch`, `WebFetch` | 搜网、抓取解析页面 |
| 发现 | `ToolSearch` | 按需动态查找加载 tools（见 §12.4） |
| 编排 | `Agent`, `Skill`, `AskUserQuestion`, `TaskCreate`, `TaskUpdate` | 派子 agent、调 skill、问用户、跟踪任务 |

Claude 只负责决定"想调哪个 tool"，放不放行由权限系统另行裁决：`allowed_tools` 列入即自动放行，`disallowed_tools` 一票否决，两边都没覆盖的按 `permission_mode` 的兜底行为处理（详见 §9「权限系统与 Human-in-the-Loop」）。tool call 被拒时，Claude 收到的 tool 结果是一条拒绝消息，通常会换一条路，或者报告任务无法继续。

并行规则：一个 turn 内的多个 tool call，只读 tools（`Read`/`Glob`/`Grep`/标记 read-only 的 MCP tools）可并发；改状态的 tools（`Edit`/`Write`/`Bash`）串行。**自定义 tool 默认按串行处理，要参与并行需在 annotations 里标 `readOnlyHint=True`（见 §10.3「Annotations 与并行」）。**

### 6.3 Context window：什么在吃你的上下文

API 是无状态的：每次请求，SDK 都要把 Claude 需要的全部内容重新发一遍。这些内容分两部分：

- **固定部分**——system prompt、CLAUDE.md、tool 定义。SDK 每次构造请求时重新拼上去，内容不随对话变化；也正因为跨请求不变，这部分自动命中 prompt caching——只有首次请求付全价，后续请求读缓存，成本和延迟都低得多。
- **累积部分**——对话历史。每个 turn 产生的 `AssistantMessage`、tool 结果都追加进去，随 turn 越滚越大，**不会在 turn 之间重置**。

各项内容逐项看：

| 来源 | 属于哪部分 | 说明 |
|---|---|---|
| System prompt | 固定 | 每次请求拼入；开销小且 prompt cache 命中 |
| Tool 定义 | 固定 | 内置 tool schema 每次都在；MCP schema 默认延迟加载，tool search 把它挡在外面 |
| CLAUDE.md | 固定 | session 开始时从磁盘读一次（经 `setting_sources`），之后全文进每次请求。**session 中途改文件不生效，要开新 session 才重新读** |
| Skill 描述 | 固定 | session 开始只读简短摘要，调用该 skill 时才加载全文 |
| 对话历史 | 累积 | 每 turn 增长；读一个大文件单次就能吃掉数千 token |

context 接近上限时 SDK **自动 compaction**：把旧的对话历史总结成摘要替换掉，流里出现 `subtype="compact_boundary"` 的 `SystemMessage`。**compaction 只压缩累积部分**——固定部分每次由 SDK 重新拼装，压缩碰不到它。

> [!warning] Compaction 会丢早期指令
> 对话里说过的指令属于对话历史，压缩时可能被摘要吞掉；写在 CLAUDE.md 里的规则属于固定部分，永远原文在场。**所以必须持久生效的规则放 CLAUDE.md，不要只在对话里说一遍。** 补救手段：CLAUDE.md 里写一节 "Summary instructions" 指定压缩时必须保留什么；`PreCompact` hook 可在压缩前归档全文；把 `/compact` 作为 prompt 发送可手动触发压缩。

CLAUDE.md 压缩指令的官方示例——节标题不是魔法字符串，压缩器按语义识别意图：

```markdown
# Summary instructions

When summarizing this conversation, always preserve:
- The current task objective and acceptance criteria
- File paths that have been read or modified
- Test results and error messages
- Decisions made and the reasoning behind them
```

保持 context 高效的四个手段：子任务交给子 agent（父级只收最终摘要，见 §14「子 agent」）、按 `AgentDefinition.tools` 精简工具集、留意 MCP schema 成本、常规任务用低 `effort`。

### 6.4 控制循环：轮次、预算与 effort

循环的四条缰绳都在 `ClaudeAgentOptions` 上：

| 选项 | 控制什么 | 默认 |
|---|---|---|
| `max_turns` | 最多 tool-use 轮次；触限收尾为 `error_max_turns` | 无限制 |
| `max_budget_usd` | 成本估算达到即停；触限收尾为 `error_max_budget_usd` | 无限制 |
| `effort` | 每轮推理深度（详下） | 不设，用模型默认 |
| `model` | 不设时用 Claude Code 默认（取决于认证方式与订阅）；显式设置可钉住版本或换小模型 | — |

`max_turns` 的计数口径：只数**带 tool call** 的 turn，第 5 步纯文本收尾那轮不占额度——`max_turns=3` 的含义是"最多让它动 3 次工具"，不是"最多 3 条回复"。注意 `ResultMessage.num_turns` 与它口径不一致：§4.2「两种输入模式」的触限实验里 `max_turns=1` 收到的却是 `num_turns=2`——`num_turns` 是事后统计的往返数，不要拿它反推限流是否精确生效。

`effort` 用延迟和 token 换推理深度。它和 extended thinking 是两件互不影响的事，可任意组合——effort 管推理深度，extended thinking 管输出里是否出现可见的思考块：

| 档位 | 行为 | 适用 |
|---|---|---|
| `"low"` | 最少推理，响应快 | 查文件、列目录 |
| `"medium"` | 均衡 | 常规编辑 |
| `"high"` | 深入分析 | 重构、调试 |
| `"xhigh"` | 更深推理 | 编码与 agent 任务（Fable 5 / Opus 4.7+ / Sonnet 5 上推荐） |
| `"max"` | 最深 | 需要深度分析的多步难题 |

两个补充口径：并非所有模型都支持 `effort` 参数；顶层 options 上的 `effort` 对整个 session 生效，子 agent 可用 `AgentDefinition.effort` 单独覆盖（见 §14「子 agent」）。

整章收官——官方的整合示例，把 allowed_tools、setting_sources、轮次上限、effort、result 分支处理放进同一个 agent：

In [1]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage


async def run_agent():
    session_id = None
    try:
        async for message in query(
            prompt="Find and fix the bug causing test failures in the auth module",
            options=ClaudeAgentOptions(
                allowed_tools=[
                    "Read",
                    "Edit",
                    "Bash",
                    "Glob",
                    "Grep",
                ],  # 列入即自动批准
                setting_sources=[
                    "project"
                ],  # 加载当前目录的 CLAUDE.md / skills / hooks
                max_turns=30,  # 防失控
                effort="high",  # 复杂调试用深推理
            ),
        ):
            if isinstance(message, ResultMessage):
                session_id = message.session_id  # 存下来，触限后可 resume 续跑
                if message.subtype == "success":
                    print("Done:", message.result)
                elif message.subtype == "error_max_turns":
                    print(f"Hit turn limit. Resume session {session_id} to continue.")
                elif message.subtype == "error_max_budget_usd":
                    print("Hit budget limit.")
                else:
                    print("Stopped:", message.subtype)
                if message.total_cost_usd is not None:  # 错误路径上可能是 None
                    print(f"Cost: ${message.total_cost_usd:.4f}")
    except Exception as error:
        # 单次 query() 在 yield 错误 result 之后还会 raise（§4.2「两种输入模式」），要接住
        print("Session ended with an error:", error)


await run_agent()

Done: 您希望我调查哪个项目的 auth 模块测试失败问题?我发现了:

1. **sigma-agent** (Go): 所有测试已通过 ✓
2. **plaud-api** (Python): 无法运行测试,缺少依赖(redis)

如果您最近遇到测试失败,能否提供以下信息:
- 哪个项目/仓库?
- 错误信息或失败的测试名称?
- 或者让我检查 CI/最近的测试日志?

另外,如果是 plaud-api,我可以先安装依赖再运行测试,但需要确认这是您关注的项目。
Cost: $0.2751
